Carga de datos al origen

In [2]:
import pandas as pd
import pyodbc

#----------------------------------- Configura la conexión a SQL Server --------------------------------
try:
    conn = pyodbc.connect(
        'DRIVER={ODBC Driver 17 for SQL Server};'
        'SERVER=LAPTOP-UJBMED9B\SQLEXPRESS;'
        'DATABASE=proyecto_origen;'
        'UID=sa;'
        'PWD=1234567890'
    )
    print("✓ Conexión a origen exitosa")

except pyodbc.Error as e:
    print(f"Error de conexión al origen: {e}")

cursor = conn.cursor()

#----------------------------------- CREAR TABLA ORIGEN --------------------------------
cursor.execute("""
IF OBJECT_ID('accidents_raw', 'U') IS NOT NULL
    DROP TABLE accidents_raw;

CREATE TABLE accidents_raw (
    ID VARCHAR(100),
    Source VARCHAR(100),
    Severity VARCHAR(50),
    Start_Time VARCHAR(50),
    End_Time VARCHAR(50),
    Start_Lat VARCHAR(50),
    Start_Lng VARCHAR(50),
    End_Lat VARCHAR(50),
    End_Lng VARCHAR(50),
    [Distance(mi)] VARCHAR(50),
    Description VARCHAR(MAX),
    Street VARCHAR(255),
    City VARCHAR(100),
    County VARCHAR(100),
    State VARCHAR(50),
    Zipcode VARCHAR(20),
    Country VARCHAR(50),
    Timezone VARCHAR(50),
    Airport_Code VARCHAR(20),
    Weather_Timestamp VARCHAR(50),
    [Temperature(F)] VARCHAR(50),
    [Wind_Chill(F)] VARCHAR(50),
    [Humidity(%)] VARCHAR(50),
    [Pressure(in)] VARCHAR(50),
    [Visibility(mi)] VARCHAR(50),
    Wind_Direction VARCHAR(50),
    [Wind_Speed(mph)] VARCHAR(50),
    [Precipitation(in)] VARCHAR(50),
    Weather_Condition VARCHAR(100),
    Amenity VARCHAR(10),
    Bump VARCHAR(10),
    Crossing VARCHAR(10),
    Give_Way VARCHAR(10),
    Junction VARCHAR(10),
    No_Exit VARCHAR(10),
    Railway VARCHAR(10),
    Roundabout VARCHAR(10),
    Station VARCHAR(10),
    Stop VARCHAR(10),
    Traffic_Calming VARCHAR(10),
    Traffic_Signal VARCHAR(10),
    Turning_Loop VARCHAR(10),
    Sunrise_Sunset VARCHAR(20),
    Civil_Twilight VARCHAR(20),
    Nautical_Twilight VARCHAR(20),
    Astronomical_Twilight VARCHAR(20)
);
""")
conn.commit()

print("Tabla origen lista")

#----------------------------------- INSERTAR DATOS --------------------------------
ruta_csv = r'C:/data/US_Accidents_March23.csv'

cursor.execute(f"""
BULK INSERT accidents_raw
FROM '{ruta_csv}'
WITH (
    FIRSTROW = 2,
    FIELDTERMINATOR = ',',
    ROWTERMINATOR = '0x0a',
    TABLOCK,
    CODEPAGE = '65001',
    FORMAT = 'CSV'
);
""")

conn.commit()

print("Carga de datos a sql lista")

cursor.close()
conn.close()

✓ Conexión a origen exitosa
Tabla origen lista
Carga de datos a sql lista


Carga de datos en destino (modelamiento dimensional)

In [3]:
import pyodbc
import pandas as pd


# --------------------------- CONEXIONES ---------------------------

def get_connection_origen():
    try:
        conn = pyodbc.connect(
            'DRIVER={ODBC Driver 17 for SQL Server};'
            'SERVER=LAPTOP-UJBMED9B\\SQLEXPRESS;'
            'DATABASE=proyecto_origen;'
            'UID=sa;'
            'PWD=1234567890',
            autocommit=False
        )
        print("Conexión a origen exitosa")
        return conn
    except pyodbc.Error as e:
        print(e)
        return None


def get_connection_destino():
    try:
        conn = pyodbc.connect(
            'DRIVER={ODBC Driver 17 for SQL Server};'
            'SERVER=LAPTOP-UJBMED9B\\SQLEXPRESS;'
            'DATABASE=proyecto_destino;'
            'UID=sa;'
            'PWD=1234567890',
            autocommit=False
        )
        print("Conexión a destino exitosa")
        return conn
    except pyodbc.Error as e:
        print(e)
        return None


# --------------------------- CREAR ESQUEMA ---------------------------

def create_datamart_schema(cursor):
    cursor.execute("""
    IF OBJECT_ID('FactAccidente','U')    IS NOT NULL DROP TABLE FactAccidente;
    IF OBJECT_ID('DimSeverity','U')      IS NOT NULL DROP TABLE DimSeverity;
    IF OBJECT_ID('DimFecha','U')         IS NOT NULL DROP TABLE DimFecha;
    IF OBJECT_ID('DimUbicacionGeo','U')  IS NOT NULL DROP TABLE DimUbicacionGeo;
    IF OBJECT_ID('DimLugar','U')         IS NOT NULL DROP TABLE DimLugar;
    IF OBJECT_ID('DimClima','U')         IS NOT NULL DROP TABLE DimClima;
    IF OBJECT_ID('DimCrossing','U')      IS NOT NULL DROP TABLE DimCrossing;
    IF OBJECT_ID('DimJunction','U')      IS NOT NULL DROP TABLE DimJunction;
    IF OBJECT_ID('DimStation','U')       IS NOT NULL DROP TABLE DimStation;
    IF OBJECT_ID('DimStop','U')          IS NOT NULL DROP TABLE DimStop;
    IF OBJECT_ID('DimTraffic','U')       IS NOT NULL DROP TABLE DimTraffic;
    IF OBJECT_ID('DimCivilTwilight','U') IS NOT NULL DROP TABLE DimCivilTwilight;

    CREATE TABLE DimSeverity (
        id       INT IDENTITY(1,1) PRIMARY KEY,
        severity VARCHAR(10)
    );

    CREATE TABLE DimFecha (
        id            INT IDENTITY(1,1) PRIMARY KEY,
        start_time    VARCHAR(16),
        anio          INT,
        mes           INT,
        dia           INT,
        hora          INT,
        dia_semana    VARCHAR(20),
        es_fin_semana BIT
    );

    CREATE TABLE DimUbicacionGeo (
        id        INT IDENTITY(1,1) PRIMARY KEY,
        start_lat VARCHAR(50),
        start_lng VARCHAR(50)
    );
    CREATE TABLE DimLugar (
        id       INT IDENTITY(1,1) PRIMARY KEY,
        street   VARCHAR(255),
        city     VARCHAR(100),
        county   VARCHAR(100),
        state    VARCHAR(50),
        zipcode  VARCHAR(20),
        timezone VARCHAR(50)
    );
    CREATE TABLE DimClima (
        id                INT IDENTITY(1,1) PRIMARY KEY,
        weather_condition VARCHAR(100)
    );
    CREATE TABLE DimCrossing (
        id       INT IDENTITY(1,1) PRIMARY KEY,
        crossing VARCHAR(10)
    );
    CREATE TABLE DimJunction (
        id       INT IDENTITY(1,1) PRIMARY KEY,
        junction VARCHAR(10)
    );
    CREATE TABLE DimStation (
        id      INT IDENTITY(1,1) PRIMARY KEY,
        station VARCHAR(10)
    );
    CREATE TABLE DimStop (
        id   INT IDENTITY(1,1) PRIMARY KEY,
        stop VARCHAR(10)
    );
    CREATE TABLE DimTraffic (
        id             INT IDENTITY(1,1) PRIMARY KEY,
        traffic_signal VARCHAR(10)
    );
    CREATE TABLE DimCivilTwilight (
        id             INT IDENTITY(1,1) PRIMARY KEY,
        civil_twilight VARCHAR(20)
    );

    CREATE TABLE FactAccidente (
        id              VARCHAR(100) PRIMARY KEY,
        severityID      INT NOT NULL,
        fechaID         INT NOT NULL,
        ubicacionGeoID  INT NOT NULL,
        lugarID         INT NOT NULL,
        climaID         INT NOT NULL,
        crossingID      INT NOT NULL,
        junctionID      INT NOT NULL,
        stationID       INT NOT NULL,
        stopID          INT NOT NULL,
        trafficID       INT NOT NULL,
        civilTwilightID INT NOT NULL,
        duracion_minutos INT,
        distance        VARCHAR(50),
        temperature     VARCHAR(50),
        wind_chill      VARCHAR(50),
        humidity        VARCHAR(50),
        pressure        VARCHAR(50),
        visibility      VARCHAR(50),
        wind_speed      VARCHAR(50),
        precipitation   VARCHAR(50),
        FOREIGN KEY (severityID)      REFERENCES DimSeverity(id),
        FOREIGN KEY (fechaID)         REFERENCES DimFecha(id),
        FOREIGN KEY (ubicacionGeoID)  REFERENCES DimUbicacionGeo(id),
        FOREIGN KEY (lugarID)         REFERENCES DimLugar(id),
        FOREIGN KEY (climaID)         REFERENCES DimClima(id),
        FOREIGN KEY (crossingID)      REFERENCES DimCrossing(id),
        FOREIGN KEY (junctionID)      REFERENCES DimJunction(id),
        FOREIGN KEY (stationID)       REFERENCES DimStation(id),
        FOREIGN KEY (stopID)          REFERENCES DimStop(id),
        FOREIGN KEY (trafficID)       REFERENCES DimTraffic(id),
        FOREIGN KEY (civilTwilightID) REFERENCES DimCivilTwilight(id)
    );
    """)
    cursor.commit()
    print("Esquema creado")


def create_indexes(cursor):
    cursor.execute("""
    CREATE INDEX IX_Fact_severityID     ON FactAccidente(severityID);
    CREATE INDEX IX_Fact_fechaID        ON FactAccidente(fechaID);
    CREATE INDEX IX_Fact_lugarID        ON FactAccidente(lugarID);
    CREATE INDEX IX_Fact_climaID        ON FactAccidente(climaID);
    CREATE INDEX IX_Fact_ubicacionGeoID ON FactAccidente(ubicacionGeoID);
    CREATE INDEX IX_DimLugar_city_state ON DimLugar(city, state);
    CREATE INDEX IX_DimFecha_time       ON DimFecha(start_time);
    CREATE INDEX IX_DimFecha_anio_mes   ON DimFecha(anio, mes);
    CREATE INDEX IX_DimFecha_hora       ON DimFecha(hora);
    """)
    cursor.commit()
    print("Índices creados")


# --------------------------- LIMPIAR DW ---------------------------

def clean_datamart(cursor):
    print("Limpiando DataMart...")
    cursor.execute("DELETE FROM FactAccidente")
    cursor.execute("DELETE FROM DimSeverity")
    cursor.execute("DELETE FROM DimFecha")
    cursor.execute("DELETE FROM DimUbicacionGeo")
    cursor.execute("DELETE FROM DimLugar")
    cursor.execute("DELETE FROM DimClima")
    cursor.execute("DELETE FROM DimCrossing")
    cursor.execute("DELETE FROM DimJunction")
    cursor.execute("DELETE FROM DimStation")
    cursor.execute("DELETE FROM DimStop")
    cursor.execute("DELETE FROM DimTraffic")
    cursor.execute("DELETE FROM DimCivilTwilight")
    cursor.commit()
    print("DataMart limpio")


# --------------------------- DIMENSIONES ---------------------------

def load_dim_severity(source_conn, dest_cursor):
    print("Cargando DimSeverity...")
    df = pd.read_sql("""
        SELECT DISTINCT CAST(Severity AS VARCHAR(10)) AS severity
        FROM accidents_raw
        WHERE Severity IS NOT NULL
    """, source_conn)
    df = df.fillna('')
    dest_cursor.fast_executemany = True
    dest_cursor.executemany(
        "INSERT INTO DimSeverity (severity) VALUES (?)",
        df.values.tolist()
    )
    dest_cursor.commit()
    print(f"  {len(df):,} registros en DimSeverity")


def load_dim_fecha(source_conn, dest_cursor):
    print("Cargando DimFecha...")
    df = pd.read_sql("""
        SELECT DISTINCT
            CONVERT(VARCHAR(16), Start_Time, 120) AS start_time
        FROM accidents_raw
        WHERE Start_Time IS NOT NULL
    """, source_conn)

    df['start_time'] = df['start_time'].fillna('')

    dt = pd.to_datetime(df['start_time'], format='%Y-%m-%d %H:%M', errors='coerce')
    df['anio']          = dt.dt.year.fillna(0).astype(int)
    df['mes']           = dt.dt.month.fillna(0).astype(int)
    df['dia']           = dt.dt.day.fillna(0).astype(int)
    df['hora']          = dt.dt.hour.fillna(0).astype(int)
    df['dia_semana']    = dt.dt.day_name().fillna('')
    df['es_fin_semana'] = dt.dt.dayofweek.isin([5, 6]).astype(int)

    dest_cursor.fast_executemany = True
    dest_cursor.executemany(
        """INSERT INTO DimFecha
               (start_time, anio, mes, dia, hora, dia_semana, es_fin_semana)
           VALUES (?,?,?,?,?,?,?)""",
        df[['start_time', 'anio', 'mes', 'dia', 'hora',
            'dia_semana', 'es_fin_semana']].values.tolist()
    )
    dest_cursor.commit()
    print(f"  {len(df):,} registros en DimFecha")


def load_dim_ubicacion_geo(source_conn, dest_cursor):
    print("Cargando DimUbicacionGeo...")
    df = pd.read_sql("""
        SELECT DISTINCT
            CAST(Start_Lat AS VARCHAR(50)) AS start_lat,
            CAST(Start_Lng AS VARCHAR(50)) AS start_lng
        FROM accidents_raw
        WHERE Start_Lat IS NOT NULL AND Start_Lng IS NOT NULL
    """, source_conn)
    df = df.fillna('')
    dest_cursor.fast_executemany = True
    dest_cursor.executemany(
        "INSERT INTO DimUbicacionGeo (start_lat, start_lng) VALUES (?,?)",
        df.values.tolist()
    )
    dest_cursor.commit()
    print(f"  {len(df):,} registros en DimUbicacionGeo")


def load_dim_lugar(source_conn, dest_cursor):
    print("Cargando DimLugar...")
    df = pd.read_sql("""
        SELECT DISTINCT
            Street, City, County, State, Zipcode, Timezone
        FROM accidents_raw
        WHERE City IS NOT NULL AND State IS NOT NULL
    """, source_conn)
    df = df.fillna('')
    dest_cursor.fast_executemany = True
    dest_cursor.executemany(
        "INSERT INTO DimLugar (street, city, county, state, zipcode, timezone) VALUES (?,?,?,?,?,?)",
        df.values.tolist()
    )
    dest_cursor.commit()
    print(f"  {len(df):,} registros en DimLugar")


def load_dim_clima(source_conn, dest_cursor):
    print("Cargando DimClima...")
    df = pd.read_sql("""
        SELECT DISTINCT Weather_Condition AS weather_condition
        FROM accidents_raw
        WHERE Weather_Condition IS NOT NULL
    """, source_conn)
    df = df.fillna('')
    dest_cursor.fast_executemany = True
    dest_cursor.executemany(
        "INSERT INTO DimClima (weather_condition) VALUES (?)",
        df.values.tolist()
    )
    dest_cursor.commit()
    print(f"  {len(df):,} registros en DimClima")


def _load_single_col_dim(source_conn, dest_cursor, col_src, table, col_dest):
    """Genérico para dimensiones de una sola columna booleana/categórica."""
    print(f"Cargando {table}...")
    df = pd.read_sql(f"""
        SELECT DISTINCT CAST({col_src} AS VARCHAR(10)) AS val
        FROM accidents_raw
        WHERE {col_src} IS NOT NULL
    """, source_conn)
    df = df.fillna('')
    dest_cursor.fast_executemany = True
    dest_cursor.executemany(
        f"INSERT INTO {table} ({col_dest}) VALUES (?)",
        df.values.tolist()
    )
    dest_cursor.commit()
    print(f"  {len(df):,} registros en {table}")


def load_dim_civil_twilight(source_conn, dest_cursor):
    print("Cargando DimCivilTwilight...")
    df = pd.read_sql("""
        SELECT DISTINCT Civil_Twilight AS civil_twilight
        FROM accidents_raw
        WHERE Civil_Twilight IS NOT NULL
    """, source_conn)
    df = df.fillna('')
    dest_cursor.fast_executemany = True
    dest_cursor.executemany(
        "INSERT INTO DimCivilTwilight (civil_twilight) VALUES (?)",
        df.values.tolist()
    )
    dest_cursor.commit()
    print(f"  {len(df):,} registros en DimCivilTwilight")


# --------------------------- MAPEOS ---------------------------

def build_maps(dest_conn):
    print("Construyendo mapeos en RAM...")

    df = pd.read_sql("SELECT id, severity FROM DimSeverity", dest_conn).fillna('')
    map_severity = {r.severity: r.id for r in df.itertuples(index=False)}

    df = pd.read_sql("SELECT id, start_time FROM DimFecha", dest_conn).fillna('')
    map_fecha = {r.start_time: r.id for r in df.itertuples(index=False)}

    df = pd.read_sql("SELECT id, start_lat, start_lng FROM DimUbicacionGeo", dest_conn).fillna('')
    map_geo = {(r.start_lat, r.start_lng): r.id for r in df.itertuples(index=False)}

    df = pd.read_sql("SELECT id, street, city, county, state FROM DimLugar", dest_conn).fillna('')
    map_lugar = {(r.street, r.city, r.county, r.state): r.id for r in df.itertuples(index=False)}

    df = pd.read_sql("SELECT id, weather_condition FROM DimClima", dest_conn).fillna('')
    map_clima = {r.weather_condition: r.id for r in df.itertuples(index=False)}

    df = pd.read_sql("SELECT id, crossing FROM DimCrossing", dest_conn).fillna('')
    map_crossing = {r.crossing: r.id for r in df.itertuples(index=False)}

    df = pd.read_sql("SELECT id, junction FROM DimJunction", dest_conn).fillna('')
    map_junction = {r.junction: r.id for r in df.itertuples(index=False)}

    df = pd.read_sql("SELECT id, station FROM DimStation", dest_conn).fillna('')
    map_station = {r.station: r.id for r in df.itertuples(index=False)}

    df = pd.read_sql("SELECT id, stop FROM DimStop", dest_conn).fillna('')
    map_stop = {r.stop: r.id for r in df.itertuples(index=False)}

    df = pd.read_sql("SELECT id, traffic_signal FROM DimTraffic", dest_conn).fillna('')
    map_traffic = {r.traffic_signal: r.id for r in df.itertuples(index=False)}

    df = pd.read_sql("SELECT id, civil_twilight FROM DimCivilTwilight", dest_conn).fillna('')
    map_twilight = {r.civil_twilight: r.id for r in df.itertuples(index=False)}

    print("  Mapeos listos")
    return (map_severity, map_fecha, map_geo, map_lugar, map_clima,
            map_crossing, map_junction, map_station, map_stop,
            map_traffic, map_twilight)


# --------------------------- FACT TABLE ---------------------------

def load_fact(source_conn, dest_cursor,
              map_severity, map_fecha, map_geo, map_lugar, map_clima,
              map_crossing, map_junction, map_station, map_stop,
              map_traffic, map_twilight):

    print("Cargando FactAccidente...")

    query = """
        SELECT
            ID,
            CAST(Severity AS VARCHAR(10))            AS severity,
            CONVERT(VARCHAR(16), Start_Time, 120)    AS start_time,
            CAST(Start_Lat AS VARCHAR(50))           AS start_lat,
            CAST(Start_Lng AS VARCHAR(50))           AS start_lng,
            Street, City, County, State,
            Weather_Condition,
            CAST(Crossing AS VARCHAR(10))            AS crossing,
            CAST(Junction AS VARCHAR(10))            AS junction,
            CAST(Station AS VARCHAR(10))             AS station,
            CAST(Stop AS VARCHAR(10))                AS stop,
            CAST(Traffic_Signal AS VARCHAR(10))      AS traffic_signal,
            Civil_Twilight,
            DATEDIFF(
                MINUTE,
                TRY_CAST(Start_Time AS DATETIME),
                TRY_CAST(End_Time   AS DATETIME)
            )                                        AS duracion_minutos,
            CAST([Distance(mi)]      AS VARCHAR(50)) AS distance,
            CAST([Temperature(F)]    AS VARCHAR(50)) AS temperature,
            CAST([Wind_Chill(F)]     AS VARCHAR(50)) AS wind_chill,
            CAST([Humidity(%)]       AS VARCHAR(50)) AS humidity,
            CAST([Pressure(in)]      AS VARCHAR(50)) AS pressure,
            CAST([Visibility(mi)]    AS VARCHAR(50)) AS visibility,
            CAST([Wind_Speed(mph)]   AS VARCHAR(50)) AS wind_speed,
            CAST([Precipitation(in)] AS VARCHAR(50)) AS precipitation
        FROM accidents_raw
    """

    CHUNK_SIZE = 500_000
    total   = 0
    skipped = 0
    dest_cursor.fast_executemany = True

    for chunk in pd.read_sql(query, source_conn, chunksize=CHUNK_SIZE):

        chunk = chunk.dropna(subset=['start_time', 'severity', 'Weather_Condition'])
        chunk = chunk.fillna('')

        # --- Resolver FKs ---
        chunk['severityID']      = chunk['severity'].map(map_severity)
        chunk['fechaID']         = chunk['start_time'].map(map_fecha)
        chunk['ubicacionGeoID']  = list(zip(chunk['start_lat'], chunk['start_lng']))
        chunk['ubicacionGeoID']  = chunk['ubicacionGeoID'].map(map_geo)
        chunk['lugarID']         = list(zip(chunk['Street'], chunk['City'],
                                            chunk['County'], chunk['State']))
        chunk['lugarID']         = chunk['lugarID'].map(map_lugar)
        chunk['climaID']         = chunk['Weather_Condition'].map(map_clima)
        chunk['crossingID']      = chunk['crossing'].map(map_crossing)
        chunk['junctionID']      = chunk['junction'].map(map_junction)
        chunk['stationID']       = chunk['station'].map(map_station)
        chunk['stopID']          = chunk['stop'].map(map_stop)
        chunk['trafficID']       = chunk['traffic_signal'].map(map_traffic)
        chunk['civilTwilightID'] = chunk['Civil_Twilight'].map(map_twilight)

        fk_cols = ['severityID', 'fechaID', 'ubicacionGeoID', 'lugarID',
                   'climaID', 'crossingID', 'junctionID', 'stationID',
                   'stopID', 'trafficID', 'civilTwilightID']
        before   = len(chunk)
        chunk    = chunk.dropna(subset=fk_cols)
        skipped += before - len(chunk)

        chunk['duracion_minutos'] = pd.to_numeric(
            chunk['duracion_minutos'], errors='coerce'
        )

        data = [
            (
                str(r.ID),
                int(r.severityID),
                int(r.fechaID),
                int(r.ubicacionGeoID),
                int(r.lugarID),
                int(r.climaID),
                int(r.crossingID),
                int(r.junctionID),
                int(r.stationID),
                int(r.stopID),
                int(r.trafficID),
                int(r.civilTwilightID),
                None if pd.isna(r.duracion_minutos) else int(r.duracion_minutos),
                str(r.distance),
                str(r.temperature),
                str(r.wind_chill),
                str(r.humidity),
                str(r.pressure),
                str(r.visibility),
                str(r.wind_speed),
                str(r.precipitation)
            )
            for r in chunk.itertuples(index=False)
        ]

        dest_cursor.executemany("""
            INSERT INTO FactAccidente (
                id, severityID, fechaID, ubicacionGeoID, lugarID,
                climaID, crossingID, junctionID, stationID, stopID,
                trafficID, civilTwilightID,
                duracion_minutos, distance, temperature, wind_chill, humidity,
                pressure, visibility, wind_speed, precipitation
            ) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
        """, data)
        dest_cursor.commit()

        total += len(data)
        print(f"  {total:,} filas insertadas  ({skipped:,} omitidas por FK nula)...")

    print(f"  FactAccidente completa: {total:,} registros  |  omitidos: {skipped:,}")


# --------------------------- MAIN ---------------------------

if __name__ == "__main__":

    source_conn = get_connection_origen()
    dest_conn   = get_connection_destino()

    if not source_conn or not dest_conn:
        exit(1)

    dest_cursor = dest_conn.cursor()

    create_datamart_schema(dest_cursor)

    load_dim_severity(source_conn, dest_cursor)
    load_dim_fecha(source_conn, dest_cursor)
    load_dim_ubicacion_geo(source_conn, dest_cursor)
    load_dim_lugar(source_conn, dest_cursor)
    load_dim_clima(source_conn, dest_cursor)
    _load_single_col_dim(source_conn, dest_cursor, 'Crossing',       'DimCrossing', 'crossing')
    _load_single_col_dim(source_conn, dest_cursor, 'Junction',       'DimJunction', 'junction')
    _load_single_col_dim(source_conn, dest_cursor, 'Station',        'DimStation',  'station')
    _load_single_col_dim(source_conn, dest_cursor, 'Stop',           'DimStop',     'stop')
    _load_single_col_dim(source_conn, dest_cursor, 'Traffic_Signal', 'DimTraffic',  'traffic_signal')
    load_dim_civil_twilight(source_conn, dest_cursor)

    create_indexes(dest_cursor)

    maps = build_maps(dest_conn)

    load_fact(source_conn, dest_cursor, *maps)

    source_conn.close()
    dest_conn.close()
    print("\nETL completado exitosamente")

Conexión a origen exitosa
Conexión a destino exitosa
Esquema creado
Cargando DimSeverity...


C:\Users\122578\AppData\Local\Temp\ipykernel_15544\3819951132.py:196: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("""


  4 registros en DimSeverity
Cargando DimFecha...


C:\Users\122578\AppData\Local\Temp\ipykernel_15544\3819951132.py:213: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("""


  2,361,526 registros en DimFecha
Cargando DimUbicacionGeo...


...

  2 registros en DimCrossing
Cargando DimJunction...
  2 registros en DimJunction
Cargando DimStation...
  2 registros en DimStation
Cargando DimStop...
  2 registros en DimStop
Cargando DimTraffic...
  2 registros en DimTraffic
Cargando DimCivilTwilight...


C:\Users\122578\AppData\Local\Temp\ipykernel_15544\3819951132.py:316: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("""


  2 registros en DimCivilTwilight
Índices creados
Construyendo mapeos en RAM...


C:\Users\122578\AppData\Local\Temp\ipykernel_15544\3819951132.py:336: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT id, severity FROM DimSeverity", dest_conn).fillna('')
C:\Users\122578\AppData\Local\Temp\ipykernel_15544\3819951132.py:339: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT id, start_time FROM DimFecha", dest_conn).fillna('')
C:\Users\122578\AppData\Local\Temp\ipykernel_15544\3819951132.py:342: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT

  Mapeos listos
Cargando FactAccidente...
  486,371 filas insertadas  (62 omitidas por FK nula)...
  975,292 filas insertadas  (680 omitidas por FK nula)...
  1,466,744 filas insertadas  (1,711 omitidas por FK nula)...
  1,957,475 filas insertadas  (1,742 omitidas por FK nula)...
  2,449,279 filas insertadas  (1,770 omitidas por FK nula)...
  2,941,411 filas insertadas  (1,803 omitidas por FK nula)...
  3,429,016 filas insertadas  (1,835 omitidas por FK nula)...
  3,916,449 filas insertadas  (1,923 omitidas por FK nula)...
  4,401,298 filas insertadas  (5,238 omitidas por FK nula)...
  4,885,724 filas insertadas  (9,023 omitidas por FK nula)...
  5,370,362 filas insertadas  (12,790 omitidas por FK nula)...
  5,855,311 filas insertadas  (16,202 omitidas por FK nula)...
  6,341,796 filas insertadas  (18,790 omitidas por FK nula)...
  6,827,479 filas insertadas  (20,608 omitidas por FK nula)...
  7,311,498 filas insertadas  (20,925 omitidas por FK nula)...
  7,533,973 filas insertadas  (2